In [1]:
from ultralytics.data.converter import convert_coco

# Convert the Training JSON
print("Converting Train data...")
convert_coco(labels_dir='../data/ml_dataset/train/', use_segments=True)

# Convert the Validation JSON
print("Converting Val data...")
convert_coco(labels_dir='../data/ml_dataset/val/', use_segments=True)

print("Conversion complete!")

Converting Train data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/ml_dataset/train/train_coco.json: 100% ━━━━━━━━━━━━ 2394/2394 2.4Kit/s 1.0s0.0s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/coco_converted
Converting Val data...
Annotations /Users/jjwill18/Desktop/europa-chaos-blocks/data/ml_dataset/val/val_coco.json: 100% ━━━━━━━━━━━━ 300/300 3.0Kit/s 0.1s
COCO data converted successfully.
Results saved to /Users/jjwill18/Desktop/europa-chaos-blocks/notebooks/coco_converted-2
Conversion complete!


In [2]:
import yaml
import os

yolo_yaml_path = "../data/ml_dataset/chaos_blocks.yaml"

# Define the dataset structure for YOLO
yolo_config = {
    # Provide the absolute path to your dataset (safest for YOLO)
    "path": os.path.abspath("../data/ml_dataset"),
    
    # YOLO automatically looks for a 'labels' folder next to these 'images' folders
    "train": "train/images",
    "val": "val/images",
    
    # Class dictionary (0 is our only class)
    "names": {
        0: "chaos_block"
    }
}

with open(yolo_yaml_path, 'w') as f:
    yaml.dump(yolo_config, f, default_flow_style=False)

print(f"Created YOLO config file at: {yolo_yaml_path}")

Created YOLO config file at: ../data/ml_dataset/chaos_blocks.yaml


In [6]:
import os
import glob
import shutil

base_ml_dir = "../data/ml_dataset"

for split in ['train', 'val']:
    labels_target_dir = os.path.join(base_ml_dir, split, "labels")
    os.makedirs(labels_target_dir, exist_ok=True)
    
    # Search for all .txt files anywhere inside the split folder
    # We ignore classes.txt or anything that isn't an image label
    all_txts = glob.glob(os.path.join(base_ml_dir, split, "**", "*.txt"), recursive=True)
    
    moved_count = 0
    for txt_path in all_txts:
        # Ignore YOLO's classes summary file if it made one
        if os.path.basename(txt_path) == "classes.txt":
            continue
            
        # If the file isn't already directly in the target 'labels' folder, move it
        if os.path.dirname(txt_path) != os.path.abspath(labels_target_dir):
            shutil.move(txt_path, os.path.join(labels_target_dir, os.path.basename(txt_path)))
            moved_count += 1
            
    print(f"Moved {moved_count} label files to the correct {split}/labels/ folder.")

Moved 2394 label files to the correct train/labels/ folder.
Moved 300 label files to the correct val/labels/ folder.


In [1]:
import os
from ultralytics import YOLO

# Tell PyTorch to use the CPU as a fallback ONLY for the missing Mac GPU math
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# Load the model
model = YOLO('yolov8n-seg.pt')

# Train it!
results = model.train(
    data="../data/ml_dataset/chaos_blocks.yaml",
    epochs=50,
    imgsz=256,
    batch=16,
    device='mps'  # Keep this as mps!
)

Ultralytics 8.4.174 🚀 Python-3.11.16 torch-2.13.0 MPS (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../data/ml_dataset/chaos_blocks.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=None, opset=None, optimize=False, optimi

/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: index_put_with_accumulate_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1786372171945/work/aten/src/ATen/Context.cpp:190.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/Users/jjwill18/miniforge3/envs/chaos_ml_env/lib/python3.11/site-packages/torch/autograd/graph.py:979: UserWarning: scatter_reduce_mps does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for

       1/50       1.3G      3.064      4.246      2.552      1.363          0        225        256: 100% ━━━━━━━━━━━━ 191/191 3.2it/s 1:000.4sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 0% ──────────── 0/13  0.5s


NotImplementedError: The operator 'torchvision::nms' is not currently implemented for the MPS device. If you want this op to be considered for addition please comment on https://github.com/pytorch/pytorch/issues/141287 and mention use-case, that resulted in missing op as well as commit hash Unknown. As a temporary fix, you can set the environment variable `PYTORCH_ENABLE_MPS_FALLBACK=1` to use the CPU as a fallback for this op. WARNING: this will be slower than running natively on MPS.